# sanitise - Agent 1 live demo (real Ollama + Presidio + spaCy)

Agent 1 finds the sensitive information in a document **before** it is ever shown to a
language model:

1. **Rule pass** (Person A): Presidio + spaCy `en_core_web_lg` + custom regex find the
   explicit identifiers (names, ages, dates, phones, emails, IDs, addresses, relations,
   body measurements) with exact character offsets.
2. **LLM pass** (Person B): one JSON-schema-constrained call to a local model (llama3:8b
   via Ollama) finds what rules cannot: medical conditions, occupations, named places and
   **quasi-identifier combinations** (details that identify someone only together).
3. **Repair loop + merge + validation** (Person C): bad model output is retried with the
   validation error fed back; spans are merged by category precedence and every offset
   is re-checked.
4. **Scoring** (Person D): precision / recall / F1 against 20 hand-labelled documents.

**How to run:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
The first run takes about 10-15 minutes (installs + model download); after that each
document takes a few seconds.

The fixtures are synthetic (fictional people). Do not paste real patient data into a
cloud notebook - the real deployment is meant to run locally.

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown **Code source.** Leave `REPO_URL` empty to upload a zip of the repository instead
#@markdown (use this for code you have not pushed yet). Otherwise `BRANCH` of `REPO_URL` is cloned.
REPO_URL = ""  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
MODEL = "llama3:8b"  #@param ["llama3:8b", "qwen2.5:7b-instruct"]
#@markdown Also run the second model on the 20 fixtures for the model comparison (section 9).
COMPARE_BOTH_MODELS = True  #@param {type:"boolean"}

OTHER_MODEL = "qwen2.5:7b-instruct" if MODEL == "llama3:8b" else "llama3:8b"
MODELS = [MODEL, OTHER_MODEL] if COMPARE_BOTH_MODELS else [MODEL]
print("models:", MODELS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code
* **`REPO_URL` empty (default):** you are asked to upload a zip whose top folder is
  `3rd_sem_el/` (for example `3rd_sem_el_colab.zip`).
* **`REPO_URL` set:** clones `BRANCH` from GitHub, e.g.
  `https://github.com/raksha-sendhil/3rd_sem_el.git`. For a private repository add a Colab
  secret named `GITHUB_TOKEN` (key icon in the left sidebar).

In [ ]:
import os
import subprocess
import zipfile

REPO_DIR = "/content/3rd_sem_el"


def upload_zip():
    from google.colab import files

    print("Choose the zip of the repository (its top folder must be 3rd_sem_el/).")
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")


if not os.path.isdir(REPO_DIR) and not REPO_URL.strip():
    upload_zip()
elif not os.path.isdir(REPO_DIR):
    url, token = REPO_URL.strip(), None
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    if token:
        url = url.replace("https://", f"https://{token}@")
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR], capture_output=True, text=True
    )
    if clone.returncode != 0:
        print((clone.stderr or "").replace(token or "\0", "***"))
        print("Clone failed - upload a zip of the repository instead.")
        upload_zip()

%cd /content/3rd_sem_el
!git log --oneline -1 2>/dev/null || echo "(from zip)"

## 3. Install Python dependencies (includes spaCy `en_core_web_lg`, ~400 MB)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
import sys

sys.path.insert(0, "/content/3rd_sem_el/src")  # editable installs need a restart otherwise
!sanitise --version

## 4. Install and start Ollama, pull the model(s) (~4.7 GB each)

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

In [ ]:
import time
import urllib.request


def ollama_up() -> bool:
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except Exception:
        return False


def ensure_ollama():
    # Start the Ollama server if it is not running (safe to call any time).
    global ollama_proc
    if ollama_up():
        return
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("/content/ollama.log", "a"),
        stderr=subprocess.STDOUT,
        env={**os.environ, "OLLAMA_KEEP_ALIVE": "1h"},  # keep the model in GPU memory
    )
    for _ in range(60):
        if ollama_up():
            print("Ollama server is running")
            return
        time.sleep(1)
    raise RuntimeError(open("/content/ollama.log").read()[-2000:])


ensure_ollama()

for m in MODELS:
    !ollama pull {m} > /dev/null 2>&1 && echo "{m} ready"
!ollama list

## 5. Configure and warm up
The first call loads spaCy (~10 s) and the model into GPU memory (~20 s). Later calls are fast.

In [ ]:
os.environ["SANITISE_LLM_BACKEND"] = "ollama"
os.environ["SANITISE_MODEL"] = MODEL
os.environ["SANITISE_LLM_TIMEOUT_S"] = "300"

from IPython.display import HTML, Markdown, display

from sanitise.demo import analyse_sync, error_html, to_html
from sanitise.errors import PipelineError

!sanitise health

t0 = time.time()
analyse_sync("Warm-up: I am a 40 year old teacher in Pune.")
print(f"warm-up done in {time.time() - t0:.1f} s")


def show(text: str, rules_only: bool = False):
    """Run Agent 1 on `text` and display the highlighted result."""
    if not rules_only:
        ensure_ollama()
    try:
        result, trace = analyse_sync(text, rules_only=rules_only)
    except PipelineError as exc:
        display(HTML(error_html(exc)))
        return None
    display(HTML(to_html(text, result, trace)))
    return result

## 6. Try it: paste any document (up to 2000 characters)
Edit `DOC` and run the cell. Spans with a **solid** border come from the rules, **dashed**
ones from the LLM. Hover over a span to see why it was flagged.

In [ ]:
DOC = """Hello doctor, I am a 52 year old ferry captain on Havelock Island and I was
diagnosed with Meniere's disease in March 2023. My daughter Kavya says the vertigo
attacks are getting worse. I can be reached at kavya.nair@example.com or 98471 22310."""

result = show(DOC)

The raw JSON the next agents receive (`DetectorResult`, spec 8.1):

In [ ]:
print(result.model_dump_json(indent=2) if result else "no result")

### Interactive version
Pick one of the 20 test documents or type your own, then press **Detect**.

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets

try:
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:
    pass

examples = {"(type your own)": ""}
for path in sorted(Path("data/fixtures").glob("*.json")):
    examples[path.stem] = json.loads(path.read_text(encoding="utf-8"))["text"]

picker = widgets.Dropdown(options=list(examples), description="Example")
box = widgets.Textarea(
    value=examples["case_07"], layout=widgets.Layout(width="100%", height="140px")
)
rules_only = widgets.Checkbox(value=False, description="rules only (skip the LLM)")
button = widgets.Button(description="Detect", button_style="primary", icon="search")
out = widgets.Output()


def on_pick(change):
    if examples[change["new"]]:
        box.value = examples[change["new"]]


def on_click(_):
    out.clear_output()
    with out:
        print("running...")
        try:
            if not rules_only.value:
                ensure_ollama()
            result, trace = analyse_sync(box.value, rules_only=rules_only.value)
            page = to_html(box.value, result, trace)
        except PipelineError as exc:
            page = error_html(exc)
        out.clear_output()
        display(HTML(page))


picker.observe(on_pick, names="value")
button.on_click(on_click)
display(widgets.VBox([picker, box, widgets.HBox([button, rules_only]), out]))

## 7. The spec's demo command (section 6)
`case_07` is the document from the spec. This is the terminal view of the same pipeline.

In [ ]:
!sanitise detect --input data/fixtures/case_07.json --verbose

## 8. Evaluate on the 20 gold documents
Runs the detector on every fixture, scores it against the gold annotations (match rule:
overlap + same category, quasi-identifiers on category only, one-to-one), lists every
miss, and writes `docs/metrics_report.md`. Recall is the primary figure: a miss is a leak.

In [ ]:
def run_dir(model: str) -> str:
    return "/content/out/" + model.replace(":", "_").replace("/", "_")


RULES_DIR, MODEL_DIR = run_dir("rules-only"), run_dir(MODEL)
!rm -rf /content/out
!sanitise detect --input data/fixtures --rules-only --out {RULES_DIR}
!sanitise detect --input data/fixtures --model {MODEL} --out {MODEL_DIR}
!sanitise score --gold data/gold --pred {MODEL_DIR} --errors --report docs/metrics_report.md --model {MODEL}

In [ ]:
display(Markdown(Path("docs/metrics_report.md").read_text(encoding="utf-8")))

## 9. Model comparison (spec section 9: choose the model on evidence)
Rules only vs. the full pipeline with each model, on the same 20 documents.

In [ ]:
from sanitise.demo import comparison_markdown, summarise_run
from sanitise.results import load_gold

for m in MODELS[1:]:
    other_dir = run_dir(m)
    !sanitise detect --input data/fixtures --model {m} --out {other_dir}

rows = [summarise_run("rules only", run_dir("rules-only"), "data/gold")]
rows += [summarise_run(f"rules + {m}", run_dir(m), "data/gold") for m in MODELS]
gold_spans = sum(len(g.spans) for g in load_gold("data/gold"))
table = comparison_markdown(rows, gold_spans)
Path("docs/model_comparison.md").write_text(table, encoding="utf-8")
display(Markdown(table))

## 10. Test suites (optional)

In [ ]:
!python -m pytest -q -p no:cacheprovider                 # fast suite, no model
!python -m pytest -q -p no:cacheprovider -m presidio     # real Presidio + spaCy
!OLLAMA_TEST_MODEL={MODEL} python -m pytest -q -p no:cacheprovider -m ollama   # real model

## 11. HTTP API (optional)
The same pipeline behind FastAPI (spec 8.4), as the Phase 3 dashboard will call it.

In [ ]:
api_log = open("/content/api.log", "w")
api_proc = subprocess.Popen(
    ["sanitise", "serve", "--port", "8000"], stdout=api_log, stderr=subprocess.STDOUT
)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/v1/health", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open("/content/api.log").read()[-2000:])


def post(path, body):
    req = urllib.request.Request(
        "http://127.0.0.1:8000" + path,
        data=json.dumps(body).encode(),
        headers={"content-type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=600) as resp:
            status, payload = resp.status, json.load(resp)
    except urllib.error.HTTPError as err:
        status, payload = err.code, json.load(err)
    print(f"POST {path} -> {status}")
    print(json.dumps(payload, indent=2)[:1200], end="\n\n")


post("/v1/detect", {"text": DOC})
post("/v1/detect", {"text": "x" * 2001})  # 413 DOC_TOO_LONG error envelope (spec 8.5)

## 12. Download the results

In [ ]:
!cd /content && rm -f agent1_results.zip && zip -qr agent1_results.zip out 3rd_sem_el/docs/metrics_report.md 3rd_sem_el/docs/model_comparison.md
from google.colab import files

files.download("/content/agent1_results.zip")

## 13. Clean up (only when you are completely done)
**Run all** reaches this cell too, so it does nothing unless you set `STOP_SERVERS = True`.
Sections 6 and 7 keep working after Run all; if Ollama was stopped, they restart it.

In [ ]:
STOP_SERVERS = False
if STOP_SERVERS:
    for proc in ("api_proc", "ollama_proc"):
        if proc in globals():
            globals()[proc].terminate()
    print("servers stopped")
else:
    print("servers left running - sections 6 and 7 can still be used")